In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load Customer CDC Work
# ============================================================

customer_cdc_df = spark.table(
    "customer360_dev.control.customer_cdc_work"
)

gold_customer_df = spark.table(
    "customer360_dev.gold.dim_customer"
)

print(
    "Customer CDC rows:",
    customer_cdc_df.count()
)

Customer CDC rows: 0


In [0]:
run_id_row = (
    customer_cdc_df
    .select("_pipeline_run_id")
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .first()
)

run_id = (
    run_id_row["_pipeline_run_id"]
    if run_id_row
    else None
)

print(
    "SCD2 Processing Run ID:",
    run_id
)

SCD2 Processing Run ID: None


In [0]:
scd2_candidate_df = (
    customer_cdc_df
    .filter(
        F.col("cdc_action").isin(
            "INSERT",
            "UPDATE"
        )
    )
)

print(
    "SCD2 candidate customers:",
    scd2_candidate_df.count()
)

SCD2 candidate customers: 0


In [0]:
current_gold_df = (
    gold_customer_df
    .filter(
        F.col("is_current") == True
    )
)

In [0]:
comparison_df = (
    scd2_candidate_df.alias("src")
    .join(
        current_gold_df.alias("tgt"),

        F.col("src.customer_id")
        ==
        F.col("tgt.customer_id"),

        how="left"
    )
)

In [0]:
scd2_change_condition = (

    ~F.col("src.first_name").eqNullSafe(
        F.col("tgt.first_name")
    )

    |

    ~F.col("src.last_name").eqNullSafe(
        F.col("tgt.last_name")
    )

    |

    ~F.col("src.email").eqNullSafe(
        F.col("tgt.email")
    )

    |

    ~F.col("src.phone_number").eqNullSafe(
        F.col("tgt.phone_number")
    )

    |

    ~F.col("src.date_of_birth").eqNullSafe(
        F.col("tgt.date_of_birth")
    )

    |

    ~F.col("src.gender").eqNullSafe(
        F.col("tgt.gender")
    )

    |

    ~F.col("src.customer_status").eqNullSafe(
        F.col("tgt.customer_status")
    )

    |

    ~F.col("src.loyalty_tier").eqNullSafe(
        F.col("tgt.loyalty_tier")
    )

    |

    ~F.col("src.preferred_language").eqNullSafe(
        F.col("tgt.preferred_language")
    )
)

In [0]:
scd2_classified_df = (
    comparison_df

    .withColumn(
        "scd2_action",

        F.when(
            F.col("tgt.customer_id").isNull(),
            F.lit("NEW_CUSTOMER")
        )

        .when(
            scd2_change_condition,
            F.lit("TYPE2_CHANGE")
        )

        .otherwise(
            F.lit("NO_SCD2_CHANGE")
        )
    )
)

In [0]:
customer_scd2_work_df = (
    scd2_classified_df

    .select(

        F.col("src.customer_id").alias(
            "customer_id"
        ),

        F.col("src.first_name").alias(
            "first_name"
        ),

        F.col("src.last_name").alias(
            "last_name"
        ),

        F.col("src.email").alias(
            "email"
        ),

        F.col("src.phone_number").alias(
            "phone_number"
        ),

        F.col("src.date_of_birth").alias(
            "date_of_birth"
        ),

        F.col("src.gender").alias(
            "gender"
        ),

        F.col("src.signup_date").alias(
            "signup_date"
        ),

        F.col("src.customer_status").alias(
            "customer_status"
        ),

        F.col("src.loyalty_tier").alias(
            "loyalty_tier"
        ),

        F.col("src.preferred_language").alias(
            "preferred_language"
        ),

        F.col("src.updated_at").alias(
            "source_updated_at"
        ),

        F.col("src._source_file").alias(
            "_source_file"
        ),

        F.col("src._source_system").alias(
            "_source_system"
        ),

        F.col("src._pipeline_run_id").alias(
            "_pipeline_run_id"
        ),

        F.col("src.cdc_action").alias(
            "cdc_action"
        ),

        F.col("scd2_action")
    )
)

In [0]:
(
    customer_scd2_work_df
    .groupBy("scd2_action")
    .count()
    .orderBy("scd2_action")
    .show()
)

+-----------+-----+
|scd2_action|count|
+-----------+-----+
+-----------+-----+



In [0]:
(
    customer_scd2_work_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "customer360_dev.control.customer_scd2_work"
    )
)

In [0]:
display(
    spark.table(
        "customer360_dev.control.customer_scd2_work"
    )
    .select(
        "customer_id",
        "loyalty_tier",
        "customer_status",
        "cdc_action",
        "scd2_action",
        "_pipeline_run_id"
    )
    .limit(50)
)

customer_id,loyalty_tier,customer_status,cdc_action,scd2_action,_pipeline_run_id
